# CCLM + CIRA Colab Notebook

Run everything directly in notebook cells (no clone/install cells).

Goal: compare CIRA method accuracy vs baseline accuracy.

## Step 1: Paths and dataset detection

In [ ]:
from pathlib import Path
import sys

root_candidates = [Path.cwd(), Path('/content/cognitive-constraint'), Path('/content')]
ROOT = None
for c in root_candidates:
    if (c / 'src' / 'cclm_cira').exists() and (c / 'configs' / 'base.yaml').exists():
        ROOT = c
        break
if ROOT is None:
    raise FileNotFoundError('Project root not found. Open notebook from repository root.')

dataset_candidates = [
    ROOT / 'data' / 'cira_lab_dataset.json',
    Path('/content/cira_lab_dataset.json'),
    Path('/content/data/cira_lab_dataset.json')
]
DATASET_PATH = next((p for p in dataset_candidates if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError('Dataset file cira_lab_dataset.json not found.')

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
OUTPUT_DIR = ROOT / 'outputs' / 'run_colab_notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('ROOT:', ROOT)
print('DATASET:', DATASET_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Step 2: Imports

In [ ]:
import json
import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from cclm_cira.data import CIRALabDataset, build_answer_space, build_vocab, collate_fn, read_samples, split_samples
from cclm_cira.evaluate import evaluate_model
from cclm_cira.model import CIRAClassifier
from cclm_cira.utils import device_for_training, load_yaml, set_seed

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Step 3: Build data loaders

In [ ]:
cfg = load_yaml(CONFIG_PATH)
set_seed(cfg['seed'])
device = device_for_training()

samples = read_samples(DATASET_PATH)
train_s, val_s, test_s = split_samples(
    samples,
    val_ratio=cfg['train']['val_ratio'],
    test_ratio=cfg['train']['test_ratio'],
    seed=cfg['seed']
)

vocab = build_vocab(train_s + val_s + test_s)
answer_to_id, id_to_answer = build_answer_space(train_s + val_s + test_s)

train_ds = CIRALabDataset(train_s, vocab, answer_to_id)
val_ds = CIRALabDataset(val_s, vocab, answer_to_id)
test_ds = CIRALabDataset(test_s, vocab, answer_to_id)

train_loader = DataLoader(train_ds, batch_size=cfg['train']['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg['train']['batch_size'], shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg['train']['batch_size'], shuffle=False, collate_fn=collate_fn)

print({'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)})

## Step 4: Train CIRA method

In [ ]:
set_seed(cfg['seed'])
model = CIRAClassifier(
    vocab_size=vocab.size,
    num_labels=len(answer_to_id),
    embedding_dim=cfg['model']['embedding_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    dropout=cfg['model']['dropout'],
    confidence_wm_prior=cfg['model']['confidence_wm_prior'],
    confidence_lm_prior=cfg['model']['confidence_lm_prior'],
).to(device)

optimizer = AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()

history = []
best_val_acc = -1.0
best_path = OUTPUT_DIR / 'best_model.pt'

for epoch in range(1, cfg['train']['epochs'] + 1):
    model.train()
    run_loss = 0.0
    run_n = 0
    for batch in train_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(
            query_ids=batch['query_ids'],
            query_mask=batch['query_mask'],
            wm_ids=batch['wm_ids'],
            wm_mask=batch['wm_mask'],
            lm_initial_ids=batch['lm_initial_ids'],
            lm_initial_mask=batch['lm_initial_mask'],
            lm_distractor_ids=batch['lm_distractor_ids'],
            lm_distractor_mask=batch['lm_distractor_mask'],
        )
        cls_loss = criterion(out.logits, batch['labels'])
        margin = torch.relu(out.sim_dist - out.sim_wm + 0.1).mean()
        loss = cls_loss + 0.25 * margin

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['train']['gradient_clip'])
        optimizer.step()

        bs = batch['labels'].size(0)
        run_loss += loss.item() * bs
        run_n += bs

    tr_loss = run_loss / max(1, run_n)
    val = evaluate_model(model, val_loader, id_to_answer, device)
    history.append({'epoch': float(epoch), 'train_loss': tr_loss, 'val_loss': val.loss, 'val_accuracy': val.accuracy})

    if val.accuracy > best_val_acc:
        best_val_acc = val.accuracy
        torch.save({'model_state_dict': model.state_dict(), 'vocab': vocab.itos, 'answer_to_id': answer_to_id, 'config': cfg}, best_path)

    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch={epoch:03d} train_loss={tr_loss:.4f} val_loss={val.loss:.4f} val_acc={val.accuracy:.3f}")

(OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best validation accuracy:', round(best_val_acc, 4))

## Step 5: Test CIRA accuracy

In [ ]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
cira_test = evaluate_model(model, test_loader, id_to_answer, device)

cira_metrics = {
    'test_loss': cira_test.loss,
    'test_accuracy': cira_test.accuracy,
    'stale_hits': cira_test.stale_hits,
    'distractor_hits': cira_test.distractor_hits
}

(OUTPUT_DIR / 'metrics_cira.json').write_text(json.dumps(cira_metrics, indent=2), encoding='utf-8')
print(json.dumps(cira_metrics, indent=2))

## Step 6: Train baseline (no interference-margin term)

In [ ]:
set_seed(cfg['seed'])
base_model = CIRAClassifier(
    vocab_size=vocab.size,
    num_labels=len(answer_to_id),
    embedding_dim=cfg['model']['embedding_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    dropout=cfg['model']['dropout'],
    confidence_wm_prior=cfg['model']['confidence_wm_prior'],
    confidence_lm_prior=cfg['model']['confidence_lm_prior'],
).to(device)

base_optim = AdamW(base_model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
base_criterion = torch.nn.CrossEntropyLoss()

best_base_val = -1.0
best_base_state = None

for epoch in range(1, cfg['train']['epochs'] + 1):
    base_model.train()
    for batch in train_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = base_model(
            query_ids=batch['query_ids'],
            query_mask=batch['query_mask'],
            wm_ids=batch['wm_ids'],
            wm_mask=batch['wm_mask'],
            lm_initial_ids=batch['lm_initial_ids'],
            lm_initial_mask=batch['lm_initial_mask'],
            lm_distractor_ids=batch['lm_distractor_ids'],
            lm_distractor_mask=batch['lm_distractor_mask'],
        )
        loss = base_criterion(out.logits, batch['labels'])

        base_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), cfg['train']['gradient_clip'])
        base_optim.step()

    base_val = evaluate_model(base_model, val_loader, id_to_answer, device)
    if base_val.accuracy > best_base_val:
        best_base_val = base_val.accuracy
        best_base_state = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}

base_model.load_state_dict(best_base_state)
base_test = evaluate_model(base_model, test_loader, id_to_answer, device)

baseline_metrics = {
    'test_loss': base_test.loss,
    'test_accuracy': base_test.accuracy,
    'stale_hits': base_test.stale_hits,
    'distractor_hits': base_test.distractor_hits
}

(OUTPUT_DIR / 'metrics_baseline.json').write_text(json.dumps(baseline_metrics, indent=2), encoding='utf-8')
print(json.dumps(baseline_metrics, indent=2))

## Step 7: Accuracy comparison (your method vs baseline)

In [ ]:
gain = cira_metrics['test_accuracy'] - baseline_metrics['test_accuracy']
comparison = {
    'cira_accuracy': cira_metrics['test_accuracy'],
    'baseline_accuracy': baseline_metrics['test_accuracy'],
    'absolute_gain': gain,
    'claim_supported': bool(gain > 0)
}

(OUTPUT_DIR / 'accuracy_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print(json.dumps(comparison, indent=2))

if gain > 0:
    print('Conclusion: Your CIRA method performs better on this split.')
elif gain == 0:
    print('Conclusion: Tie on this split.')
else:
    print('Conclusion: Baseline is better on this split.')